# Final Project — AI and Cyber
## Track A — Dataset Project: Edge-IIoTset

**Team members:** Ibrahim Hamed DICKO, Mariam ARKIK, Abdoulaye TOURE
**Course:** AI and Cyber — Sourav Rai
**Random seed used throughout:** `42`

---

This notebook follows the required Track A workflow:
1. Data Acquisition
2. Exploratory Data Analysis (EDA)
3. Model Development and Comparison
4. Hyperparameter Tuning
5. Evaluation

Each section includes the questions asked in the assignment, answered directly in markdown.

## 1. Data Acquisition

### Source
- **Dataset:** Edge-IIoTset — A New Comprehensive Realistic Cyber Security Dataset of IoT
  and IIoT Applications for Centralized and Federated Learning
- **URL:** https://www.kaggle.com/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot
- **Paper:** Ferrag, M.A., Friha, O., Hamouda, D., Maglaras, L., Janicke, H. — *IEEE Access*,
  April 2022, DOI: 10.1109/ACCESS.2022.3165809
- **File used:** `DNN-EdgeIIoT-dataset.csv` (the "Selected dataset for ML and DL" — pre-extracted
  flow features, as opposed to the raw PCAPs)

### Domain context — what does this dataset represent?

The dataset was generated from a **physical 7-layer IIoT/Edge testbed** (Cloud, Network
Function Virtualization, Blockchain, Fog, Software-Defined Networking, Edge Computing, and
IoT/IIoT Perception layers), using **more than 10 real IoT device types** (temperature/humidity
sensors, ultrasonic sensors, flame sensors, pH meters, heart-rate sensors, etc.) communicating
over a mix of **IT protocols (TCP/IP, UDP, ICMP, HTTP, DNS, ARP)** and **OT/IIoT protocols
(MQTT, Modbus/TCP)**.

Traffic was captured at the edge gateway and exported as **61 flow-level features** plus two
label columns:
- `Attack_label` — **binary** target (0 = normal, 1 = attack)
- `Attack_type` — **multiclass** target (Normal + 14 attack types)

**14 attack types, grouped into 5 categories** (as defined in the original paper):

| Category | Attack types |
|---|---|
| DoS / DDoS | DDoS_UDP, DDoS_ICMP, DDoS_TCP, DDoS_HTTP |
| Information gathering | Port_Scanning, Vulnerability_scanner, Fingerprinting |
| Man-in-the-Middle | MITM |
| Injection | SQL_injection, XSS, Uploading |
| Malware | Backdoor, Ransomware, Password |

**Why this matters for security:** unlike CLaMP (static, file-level), this is a **network
flow-level IDS dataset** — the kind of data a SOC's NIDS/SIEM would process in real time. The
attack categories map directly onto the **Cyber Kill Chain**: information gathering
(reconnaissance), injection/MITM (exploitation), backdoor/ransomware (installation & actions
on objectives). The inclusion of **MQTT and Modbus** features is what makes this dataset
specifically *industrial* IoT (IIoT) — these are SCADA/PLC protocols rarely seen in
general-purpose network IDS datasets.

### Granularity
**One row = one network flow** (an aggregated sequence of packets between two endpoints),
not a single packet. This is *flow-level*, not file-level (CLaMP) or session-level.

### ⚠️ Scale — read this before downloading

The raw `DNN-EdgeIIoT-dataset.csv` has **2,219,201 rows** and **63 columns**, split
roughly **72.8% Normal / 27.2% attack** at the raw level. This is too large for a free
Colab/laptop kernel to explore comfortably — **we must work from a stratified subset**, per
the project guide's instructions.

### Step 1 — Download the raw file (run once, locally)

```bash
pip install kaggle
# Place your kaggle.json in ~/.kaggle/ (chmod 600), then:
kaggle datasets download -d mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot \
    -f "Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv" \
    -p data/raw --unzip
```

This produces `data/raw/DNN-EdgeIIoT-dataset.csv` (~hundreds of MB — **do not commit this
file to git**, it's in `.gitignore`).

### Step 2 — Create a reproducible stratified sample (run once)

We take a **10% stratified sample on `Attack_type`** (preserves all 15 classes,
including the rare `MITM` class with only ~90 raw rows) and save it as the file this
notebook actually works with.

In [5]:
# --- Run this cell ONCE, after downloading the raw CSV (Step 1 above) ---
# It reads the ~2.2M-row raw file and writes a 10% stratified sample (~220k rows)
# that the rest of this notebook (and your teammates) will use.

import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH = "../data/raw/DNN-EdgeIIoT-dataset.csv"
SAMPLE_PATH = "../data/DNN-EdgeIIoT-sample10.csv"

df_full = pd.read_csv(RAW_PATH, low_memory=False)
df_sample, _ = train_test_split(
    df_full, train_size=0.10, stratify=df_full["Attack_type"], random_state=42
)
df_sample.to_csv(SAMPLE_PATH, index=False)
print(f"Sample written: {df_sample.shape}")

Sample written: (221920, 63)


In [6]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    precision_recall_curve, PrecisionRecallDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

print("Libraries loaded.")

Libraries loaded.


### Data loader function

The official preprocessing notebook (Ferrag et al.) drops 15 columns before modelling.
We adopt the same list, grouped here by **why** each is dropped — this grouping is itself
useful for the "operational challenges" discussion in Section 5:

| Reason for dropping | Columns |
|---|---|
| **Identity leakage** (IP addresses identify specific testbed hosts, not general attack patterns) | `ip.src_host`, `ip.dst_host`, `arp.src.proto_ipv4`, `arp.dst.proto_ipv4` |
| **Timestamps** (not generalizable; risk of temporal leakage) | `frame.time`, `icmp.transmit_timestamp` |
| **Raw payload / high-cardinality strings** (would need NLP-style processing, out of scope) | `http.file_data`, `http.request.full_uri`, `http.request.uri.query`, `tcp.options`, `tcp.payload`, `mqtt.msg` |
| **Port numbers** (testbed-specific; risk of overfitting to fixed ports rather than learning behaviour) | `tcp.srcport`, `tcp.dstport`, `udp.port` |

The function below also separates the **two targets** — `Attack_label` (binary) and
`Attack_type` (multiclass) — as required by the project ("perform both binary and
multiclass classification").

In [7]:
DROP_COLUMNS = [
    # identity leakage
    "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
    # timestamps
    "frame.time", "icmp.transmit_timestamp",
    # raw payload / very high-cardinality strings
    "http.file_data", "http.request.full_uri", "http.request.uri.query",
    "tcp.options", "tcp.payload", "mqtt.msg",
    # testbed-specific port numbers
    "tcp.srcport", "tcp.dstport", "udp.port",
]


def load_edge_iiot_data(path="../data/DNN-EdgeIIoT-sample10.csv"):
    """
    Load the Edge-IIoTset stratified sample.

    Returns
    -------
    df : full DataFrame (features + both targets), after dropping leakage columns
    X  : feature DataFrame (both target columns dropped)
    y_binary     : Attack_label (0 = normal, 1 = attack)
    y_multiclass : Attack_type  (Normal + 14 attack types)
    """
    df = pd.read_csv(path, low_memory=False)

    assert "Attack_label" in df.columns, "Expected an 'Attack_label' column"
    assert "Attack_type" in df.columns, "Expected an 'Attack_type' column"

    existing_drops = [c for c in DROP_COLUMNS if c in df.columns]
    df = df.drop(columns=existing_drops)

    X = df.drop(columns=["Attack_label", "Attack_type"])
    y_binary = df["Attack_label"]
    y_multiclass = df["Attack_type"]

    return df, X, y_binary, y_multiclass


df, X, y_binary, y_multiclass = load_edge_iiot_data()
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns "
      f"({X.shape[1]} features + 2 targets).")
print(f"\nBinary target (Attack_label) distribution:")
print(y_binary.value_counts(normalize=True).round(3))
print(f"\nMulticlass target (Attack_type) — {y_multiclass.nunique()} classes:")
print(y_multiclass.value_counts())

Loaded 221920 rows and 48 columns (46 features + 2 targets).

Binary target (Attack_label) distribution:
Attack_label
0    0.728
1    0.272
Name: proportion, dtype: float64

Multiclass target (Attack_type) — 15 classes:
Attack_type
Normal                   161564
DDoS_UDP                  12157
DDoS_ICMP                 11644
SQL_injection              5120
Password                   5015
Vulnerability_scanner      5011
DDoS_TCP                   5006
DDoS_HTTP                  4991
Uploading                  3763
Backdoor                   2486
Port_Scanning              2256
XSS                        1592
Ransomware                 1093
MITM                        122
Fingerprinting              100
Name: count, dtype: int64


In [8]:
!ls -lh ../data/

Permissions Size User  Date Modified  Name
drwxr-xr-x     - abdou 15 juin  14:46  raw
.rw-r--r--  121M abdou 15 juin  15:23  DNN-EdgeIIoT-sample10.csv
.rw-r--r--     0 abdou 15 juin  14:31 󰡯 gitkeep


---
## Roadmap — what's next

This notebook currently covers **Section 1 (Data Acquisition)**, including:
- Domain context, source, and attack-category mapping
- The mandatory 10% stratified-sampling step (run once on the raw 2.2M-row file)
- A data loader that drops the 15 leakage/high-cardinality columns and exposes **both**
  targets (`Attack_label` binary, `Attack_type` multiclass)

> ⚠️ **Before the next session:** at least one team member needs to run the Kaggle download
> + sampling step (the commented-out cell above) and commit the resulting
> `data/DNN-EdgeIIoT-sample10.csv` is **not** committed (too large) — instead, everyone
> regenerates it locally with `random_state=42`, which guarantees identical samples across
> the team.

### 2. EDA — remaining steps (2a–2i)
Same structure as before, applied to Edge-IIoTset:
- **2a. Snapshot** — `head()`, `info()`, dtypes (mix of numeric flow features + a handful
  of categorical protocol fields like `http.request.method`, `mqtt.protoname`)
- **2b. Missing values** — several columns (`icmp.unused`, `http.tls_port`, `dns.qry.type`,
  `mqtt.msg_decoded_as`) are known from prior work to be near-constant/empty — verify and
  decide drop vs. flag
- **2c. Duplicates** — flow-based data often has many duplicate rows; the official
  preprocessing drops them
- **2d. Descriptive statistics** — numeric flow stats + categorical protocol field value
  counts
- **2e. Target balance** — **both** `Attack_label` (binary) and `Attack_type` (15-class,
  heavily imbalanced — `MITM` has only ~90 raw rows / ~9 in our 10% sample)
- **2f. Correlation matrix** — many TCP/MQTT flag columns likely highly correlated
- **2g. Scaling/encoding** — StandardScaler for numeric, one-hot for the ~7 categorical
  protocol fields
- **2h. Train–test split** — stratified on `Attack_type` (covers both targets)
- **2i. Feature importance** — correlation with `Attack_label`, plus a first look at
  per-category importance for `Attack_type`

### 3–5. Models, tuning, evaluation
Same as planned for CLaMP — but now run **twice**: once for binary (`Attack_label`) and
once for multiclass (`Attack_type`), as required.

---
**Next message:** Section 2a–2i — EDA on the Edge-IIoTset sample.